# Using Financial Data Example #1: Calculating Altman Z" Score

Professor Altman first formulated his infamous "Z Score" in [1968](http://www.defaultrisk.com/_pdf6j4/Financial_Ratios_Discriminant_Anlss_n_Prdctn_o_Crprt_Bnkrptc.pdf) while at NYU. The "Z Score" attempts to quantify the likelihood that a company defaults. After several iterations, the Altman Z" (Double Prime) Score was developed to better quantify a company's credit risk. Professor Altman's presentation [here](http://pages.stern.nyu.edu/~ealtman/3-%20CopCrScoringModels.pdf) walks through several models including this one. When it was initially developed there were some crude cutoffs for the scores - above 2.6 and the firm was "healthy", between 1.1-2.6 was the "grey area", and below 1.1 and the firm as at risk of bankruptcy. However, over time that crude scale was refined. One of the unique things about the Altman Z" Score today is that we have a mapping to conventional credit ratings. Below we walk through the example of calculating the score from scratch using [Alpha Vantage data](https://www.alphavantage.co/documentation/#fundamentals) (this example originally used the IEX Cloud API, which was discontinued in August 2024).

$$ Z” = 6.56x_1 +3.26x_2 + 6.72x_3 + 1.05x_4 $$
$$ \textrm{Where:} $$
$$ x_1 = \textrm{Working Capital / Total Assets} $$
$$ x_2 = \textrm{Retained Earnings / Total Assets} $$
$$ x_3 = \textrm{EBIT / Total Assets} $$
$$ x_4 = \textrm{Market Value of Equity / Total Liabilities} $$

In [ ]:
import os
import time
import requests

ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"
# Alpha Vantage's public "demo" key works out of the box, but only serves real
# data for the "IBM" ticker used below, and throttles shared demo traffic to a
# couple of requests per second. For any other ticker (and a higher, personal
# rate limit), grab a free key at https://www.alphavantage.co/support/#api-key
# and set it as an environment variable rather than hardcoding it here:
#   export ALPHA_VANTAGE_API_KEY=your_key_here
ALPHA_VANTAGE_API_KEY = os.environ.get("ALPHA_VANTAGE_API_KEY", "demo")

def av_fundamentals(function, ticker):
    '''Fetch a fundamentals endpoint (OVERVIEW, INCOME_STATEMENT, BALANCE_SHEET) from Alpha Vantage.'''
    time.sleep(0.6)  # stay under the demo key's shared rate limit
    response = requests.get(
        ALPHA_VANTAGE_BASE_URL,
        params={"function": function, "symbol": ticker, "apikey": ALPHA_VANTAGE_API_KEY},
    )
    response.raise_for_status()
    return response.json()

In [ ]:
ticker = "IBM"  # the "demo" API key only serves real data for IBM; use your own key for other tickers
incomeStatement = av_fundamentals("INCOME_STATEMENT", ticker)["annualReports"][0]
balanceSheet = av_fundamentals("BALANCE_SHEET", ticker)["annualReports"][0]
stats = av_fundamentals("OVERVIEW", ticker)

In [ ]:
x1 = ( float(balanceSheet["totalCurrentAssets"]) - float(balanceSheet["totalCurrentLiabilities"]) ) / float(balanceSheet["totalAssets"])

In [ ]:
x2 = float(balanceSheet["retainedEarnings"]) / float(balanceSheet["totalAssets"])

In [ ]:
x3 = float(incomeStatement["ebit"]) / float(balanceSheet["totalAssets"])

In [ ]:
x4 = float(stats["MarketCapitalization"]) / float(balanceSheet["totalLiabilities"])

In [ ]:
6.56 * x1 + 3.26 * x2 + 6.72 * x3 + 1.05 * x4

In [ ]:
def altmanZDoublePrime( ticker ):
    '''
    Calculate the Altman Z" Score for a given ticker

    ticker = string, ticker to calculate the Z-score for. The Alpha Vantage
    "demo" API key (the default) only returns real data for "IBM" -- set the
    ALPHA_VANTAGE_API_KEY environment variable to your own free key
    (https://www.alphavantage.co/support/#api-key) to use other tickers.
    '''
    incomeStatement = av_fundamentals("INCOME_STATEMENT", ticker)["annualReports"][0]
    balanceSheet = av_fundamentals("BALANCE_SHEET", ticker)["annualReports"][0]
    stats = av_fundamentals("OVERVIEW", ticker)
    x1 = ( float(balanceSheet["totalCurrentAssets"]) - float(balanceSheet["totalCurrentLiabilities"]) ) / float(balanceSheet["totalAssets"])
    x2 = float(balanceSheet["retainedEarnings"]) / float(balanceSheet["totalAssets"])
    x3 = float(incomeStatement["ebit"]) / float(balanceSheet["totalAssets"])
    x4 = float(stats["MarketCapitalization"]) / float(balanceSheet["totalLiabilities"])
    return 6.56 * x1 + 3.26 * x2 + 6.72 * x3 + 1.05 * x4

In [ ]:
altmanZDoublePrime("IBM")

In [ ]:
import numpy as np

def altmanZDPImpliedRating( ticker ):
    '''
    Calculate the implied credit rating from a company's Altman Z" Score 
    
    ticker = string, user input for which to calculate the Z-score. Not case sensitive.
    '''
    adjZScore = 3.25 + altmanZDoublePrime( ticker )
    zMap = [ 8.15, 7.6, 7.3, 7., 6.85, 6.65, 6.4, 6.25, 5.85, 5.65, 5.25, 4.95, 4.75, 4.5, 4.15, 3.75, 3.2, 2.5, 1.75 ]
    scores = [ "AAA", "AA+", "AA", "AA-", "A+", "A", "A-", "BBB+", "BBB", "BBB-", "BB+", "BB", "BB-", "B+", "B", "B-", "CCC+", "CCC", "CCC-", "D" ] 
    return scores[ zMap.index( np.array( zMap )[ np.array( zMap ) < adjZScore ].max() ) ]

In [ ]:
altmanZDPImpliedRating("IBM")